In [28]:
import json
import sagemaker
import boto3

iam = boto3.client('iam')
role = iam.get_role(RoleName='sagemaker-dlc-demo')['Role']['Arn']


In [40]:
llm_image = "754289655784.dkr.ecr.us-east-1.amazonaws.com/hf-aws-dlcs/huggingface-vllm:0.16.0-transformers4.57.5-gpu-py312-cu129-ubuntu22.04"

In [41]:

import json
from sagemaker.huggingface import HuggingFaceModel

# sagemaker config
instance_type = "ml.g5.12xlarge"
health_check_timeout = 900


# Define Model and Endpoint configuration parameter
config = {
  'SM_VLLM_MODEL': "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice", # model_id from hf.co/models
  'SM_VLLM_MAX_MODEL_LEN': "2048",  # Max length of input text
  'SM_VLLM_HOST': "0.0.0.0",  # Required for SageMaker networking
  'SM_VLLM_TRUST_REMOTE_CODE': "true",  # Required to allow custom code from Hugging Face Hub
  'SM_VLLM_OMNI': "true",  # Enable Omni mode for better performance
  #'HUGGING_FACE_HUB_TOKEN': "<REPLACE WITH YOUR TOKEN>"
}


# create HuggingFaceModel with the image uri
llm_model = HuggingFaceModel(
  role=role,
  image_uri=llm_image,
  env=config,
  name="qwen3-tts-12hz-1-7b-customvoice"
)

In [42]:
# Deploy model to an endpoint
# https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model.deploy
llm = llm_model.deploy(
  initial_instance_count=1,
  instance_type=instance_type,
  container_startup_health_check_timeout=health_check_timeout,
  inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1", 
  wait=True
)


Using already existing model: qwen3-tts-12hz-1-7b-customvoice


------------------!

In [44]:
payload = {
    "task": "text-to-speech",
    "input": "Hello, how are you?",
    "voice": "vivian",
    "language": "English",
}

audio_bytes = llm.predict(payload, initial_args={"ContentType": "application/json", "Accept": "application/octet-stream"})

with open("output.wav", "wb") as f:
    f.write(audio_bytes)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │   "language": "English",                                                                  │
│    6 }                                                                                           │
│    7                                                                                             │
│ ❱  8 audio_bytes = llm.predict(payload, initial_args={"ContentType": "application/json", "Acc    │
│    9                                                                                             │
│   10 with open("output.wav", "wb") as f:                                                         │
│   11 │   f.write(audio_bytes)                                                                    │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/sagemaker/base_predictor.py:213 in predict                                                 │
│                                                                                                  │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│   212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│ ❱ 213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│   216 │   │   """Placeholder docstring"""                                                        │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/sagemaker/base_predictor.py:219 in _handle_response                                        │
│                                                                                                  │
│   216 │   │   """Placeholder docstring"""                                                        │
│   217 │   │   response_body = response["Body"]                                                   │
│   218 │   │   content_type = response.get("ContentType", "application/octet-stream")             │
│ ❱ 219 │   │   return self.deserializer.deserialize(response_body, content_type)                  │
│   220 │                                                                                          │
│   221 │   def _create_request_args(                                                              │
│   222 │   │   self,                                                                              │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/sagemaker/base_deserializers.py:277 in deserialize                                         │
│                                                                                                  │
│   274 │   │   │   object: The JSON-formatted data deserialized into a Python object.             │
│   275 │   │   """                                                                                │
│   276 │   │   try:                                                                               │
│ ❱ 277 │   │   │   return json.load(codecs.getreader("utf-8"

In [45]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

endpoint_name = "qwen3-tts-12hz-1-7b-customvoice-2026-03-18-15-39-37-242"

payload = {
    "task": "text-to-speech",
    "input": "Hello, how are you?",
    "voice": "vivian",
    "language": "English",
}

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="audio/wav",
    Body=json.dumps(payload),
)

audio_bytes = response["Body"].read()

with open("output.wav", "wb") as f:
    f.write(audio_bytes)

print("Saved output.wav")

Saved output.wav


In [ ]:
# Prompt to generate
messages=[
    { "role": "user", "content": "Give me a short introduction to large language model" }
  ]

# Generation arguments
parameters = {
    "top_p": 0.6,
    "temperature": 0.9,
    "max_tokens": 128,
}

chat = llm.predict({"messages" :messages, **parameters})

print(chat["choices"][0]["message"]["content"].strip())

In [18]:
llm.delete_model()
llm.delete_endpoint()

In [19]:
llm_model.delete_model()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 llm_model.delete_model()                                                                     │
│   2                                                                                              │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/sagemaker/model.py:1990 in delete_model                                                    │
│                                                                                                  │
│   1987 │   │   │   raise ValueError(                                                             │
│   1988 │   │   │   │   "The SageMaker model must be created first before attempting to delete."  │
│   1989 │   │   │   )                                                                             │
│ ❱ 1990 │   │   self.sagemaker_session.delete_model(self.name)                                    │
│   1991                                                                                           │
│   1992                                                                                           │
│   1993 class FrameworkModel(Model):                                                              │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/sagemaker/session.py:5589 in delete_model                                                  │
│                                                                                                  │
│   5586 │   │   │   model_name (str): Name of the Amazon SageMaker model to delete.               │
│   5587 │   │   """                                                                               │
│   5588 │   │   logger.info("Deleting model with name: %s", model_name)                           │
│ ❱ 5589 │   │   self.sagemaker_client.delete_model(ModelName=model_name)                          │
│   5590 │                                                                                         │
│   5591 │   def list_group_resources(self, group, filters, next_token: str = ""):                 │
│   5592 │   │   """To list group resources with given filters                                     │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-containers/.dlc/lib/python3.12/site-pac │
│ kages/botocore/client.py:602 in _api_call                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /Users/ehcalabres/Documents/dev/ehcalabres/deep-learning-co